In [0]:
%run ../../02_common_utils/operations

In [0]:
from pyspark.sql.functions import *

In [0]:
# ─── CONFIG ──────────────────────────────────────────────────────────────
# NOTE: Customer.txt (B2/B3) is ALREADY merged into silver.customer by the
# silver_customer_customermgmt notebook via UNION. This notebook exists as a
# STANDALONE reference / incremental notebook if you need to re-run B2 or B3
# customer CDC updates independently (e.g. after a bronze re-ingest).
#
# Source: bronze.customer   → B2: 50 rows, B3: 50 rows (cumulative = 100)
# Target: silver.customer   → MERGE on C_ID (latest wins)

catalog             = "charles_schwab_retailbrokerage_dev_team_lemma"
bronze_customer_txt = f"{catalog}.bronze.customer"
silver_customer     = f"{catalog}.silver.customer"

In [0]:
# ─── STEP 1: Read bronze.customer & register as temp view ────────────────
# Schema: CDC_FLAG, C_ID, CDC_DSN, C_TAX_ID, C_ST_ID, C_L_NAME, C_F_NAME,
#         C_M_NAME, C_GNDR, C_TIER, C_DOB, C_ADLINE1, C_ADLINE2, C_ZIPCODE,
#         C_CITY, C_STATE_PROV, C_CTRY, C_CTRY_1..3, C_AREA_1..3, C_LOCAL_1..3,
#         C_EXT_1..3, C_EMAIL_1, C_EMAIL_2, C_LCL_TX_ID, C_NAT_TX_ID,
#         _source_name, _source_file, _batch_id, _run_id, _ingest_ts

spark.read.table(bronze_customer_txt).createOrReplaceTempView("v_bronze_customer")
carried_run_id = spark.sql(f"SELECT _run_id FROM {bronze_customer_txt} ORDER BY _ingest_ts DESC LIMIT 1").first()[0]
carried_batch = spark.sql(f"SELECT _batch_id FROM {bronze_customer_txt} ORDER BY _ingest_ts DESC LIMIT 1").first()[0]

print(f"bronze.customer rows: {spark.sql('SELECT COUNT(*) FROM v_bronze_customer').first()[0]}")
print(f"carried_run_id: {carried_run_id}")
print(f"carried_batch: {carried_batch}")

In [0]:
log_pipeline_message(spark, carried_run_id, 'INFO', 'silver_customer_customer', 'Starting processing for standalone silver customer CDC updates')
start_pipeline_run(spark, carried_run_id, carried_batch)
log_domain_run_status(spark, carried_run_id, carried_batch, 'CUSTOMER', 'RUNNING')

In [0]:
# ─── STEP 2: Transform — align to silver.customer schema ─────────────────
# Key differences vs bronze.customermgmt:
#   - No ActionTS  → use _ingest_ts as action_ts for dedup ordering
#   - Email cols   → C_EMAIL_1, C_EMAIL_2 (not C_PRIM_EMAIL / C_ALT_EMAIL)
#   - _batch_id    → rename to _batch
# CDC_FLAG values: I=Insert (35/batch), U=Update (15/batch). No D in dataset.
# Strategy: ALL rows participate regardless of CDC_FLAG — latest wins by _ingest_ts

df_customer = spark.sql("""
    SELECT
        CAST(_ingest_ts      AS TIMESTAMP)   AS action_ts,      -- proxy for ActionTS
        CAST(C_ID            AS BIGINT)      AS C_ID,
        C_TAX_ID,
        C_GNDR,
        TRY_CAST(C_TIER      AS TINYINT)     AS C_TIER,
        CAST(C_DOB           AS DATE)        AS C_DOB,
        C_L_NAME, C_F_NAME, C_M_NAME,
        C_ADLINE1, C_ADLINE2, C_ZIPCODE, C_CITY, C_STATE_PROV, C_CTRY,
        C_EMAIL_1                            AS primary_email,
        C_EMAIL_2                            AS alternate_email,
        CONCAT_WS('-', NULLIF(C_CTRY_1,''), NULLIF(C_AREA_1,''), NULLIF(C_LOCAL_1,''), NULLIF(C_EXT_1,'')) AS phone1,
        CONCAT_WS('-', NULLIF(C_CTRY_2,''), NULLIF(C_AREA_2,''), NULLIF(C_LOCAL_2,''), NULLIF(C_EXT_2,'')) AS phone2,
        CONCAT_WS('-', NULLIF(C_CTRY_3,''), NULLIF(C_AREA_3,''), NULLIF(C_LOCAL_3,''), NULLIF(C_EXT_3,'')) AS phone3,
        C_LCL_TX_ID,
        C_NAT_TX_ID,
        _batch_id                            AS _batch,
        _run_id,
        current_timestamp()                  AS _load_ts
    FROM v_bronze_customer
""")

df_customer.createOrReplaceTempView("v_customer_raw")
print(f"Rows after transform: {df_customer.count()}")

In [0]:
# ─── STEP 3: Deduplicate by C_ID — latest _ingest_ts wins ────────────────
# If same customer (C_ID=U) appears in B2 and B3, B3 row wins (latest timestamp)

df_customer_dedup = spark.sql("""
    SELECT * EXCEPT (rn)
    FROM (
        SELECT *,
            ROW_NUMBER() OVER (
                PARTITION BY C_ID
                ORDER BY action_ts DESC
            ) AS rn
        FROM v_customer_raw
    )
    WHERE rn = 1
""")
# action_ts is KEPT in output so it matches the target silver.customer schema
# (silver.customer was first created by customermgmt notebook which included action_ts)

# from pyspark.sql.window import Window
# from pyspark.sql.functions import row_number
# window_cust = Window.partitionBy("C_ID").orderBy(col("action_ts").desc())
# df_customer_dedup = df_customer \
#     .withColumn("rn", row_number().over(window_cust)) \
#     .filter(col("rn") == 1).drop("rn")    # NOTE: keep action_ts for MERGE compatibility

df_customer_dedup.createOrReplaceTempView("v_customer_dedup")
source_count = df_customer_dedup.count()
print(f"Deduplicated rows to merge: {source_count}")

In [0]:
# ─── STEP 4: MERGE into silver.customer ──────────────────────────────────
# MATCHED   → UPDATE in place (15 U records per batch — count stays same)
# NOT MATCHED → INSERT    (35 I records per batch — count grows by 35)

spark.sql(f"""
    MERGE INTO {silver_customer} AS tgt
    USING v_customer_dedup       AS src
    ON tgt.C_ID = src.C_ID
    WHEN MATCHED THEN
        UPDATE SET *
    WHEN NOT MATCHED THEN
        INSERT *
""")

# from delta.tables import DeltaTable
# DeltaTable.forName(spark, silver_customer).alias("tgt") \
#     .merge(df_customer_dedup.alias("src"), "tgt.C_ID = src.C_ID") \
#     .whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

silver_customer_count = spark.read.table(silver_customer).count()
print(f"silver.customer total rows: {silver_customer_count}")
# Expected: 15,315 after B2 | 15,350 after B3

In [0]:
# ─── STEP 5: Operations Audit Logging ────────────────────────────────────
log_pipeline_recon(
    spark=spark, run_id=carried_run_id, batch_id="ALL",
    domain="CUSTOMER", table_name="customer",
    source_layer="bronze", target_layer="silver",
    source_count=source_count, target_count=silver_customer_count
)
log_audit_event(
    spark=spark, run_id=carried_run_id, batch="ALL",
    layer="silver", table_name="customer",
    operation="MERGE", rows_affected=silver_customer_count
)

null_cid_count = spark.sql("SELECT COUNT(*) FROM v_customer_dedup WHERE C_ID IS NULL").first()[0]
log_dq_result(spark, carried_run_id, "silver.customer", "Null C_ID Check", null_cid_count, source_count)

log_domain_run_status(spark, carried_run_id, carried_batch, 'CUSTOMER', 'COMPLETED')
end_pipeline_run(spark, carried_run_id, 'SUCCESS')
log_pipeline_message(spark, carried_run_id, 'INFO', 'silver_customer_customer', 'Successfully completed standalone customer CDC updates')